--- Day 24: Crossed Wires ---
You and The Historians arrive at the edge of a large grove somewhere in the jungle. After the last incident, the Elves installed a small device that monitors the fruit. While The Historians search the grove, one of them asks if you can take a look at the monitoring device; apparently, it's been malfunctioning recently.

The device seems to be trying to produce a number through some boolean logic gates. Each gate has two inputs and one output. The gates all operate on values that are either true (1) or false (0).

AND gates output 1 if both inputs are 1; if either input is 0, these gates output 0.
OR gates output 1 if one or both inputs is 1; if both inputs are 0, these gates output 0.
XOR gates output 1 if the inputs are different; if the inputs are the same, these gates output 0.
Gates wait until both inputs are received before producing output; wires can carry 0, 1 or no value at all. There are no loops; once a gate has determined its output, the output will not change until the whole system is reset. Each wire is connected to at most one gate output, but can be connected to many gate inputs.

Rather than risk getting shocked while tinkering with the live system, you write down all of the gate connections and initial wire values (your puzzle input) so you can consider them in relative safety. For example:

x00: 1
x01: 1
x02: 1
y00: 0
y01: 1
y02: 0

x00 AND y00 -> z00
x01 XOR y01 -> z01
x02 OR y02 -> z02
Because gates wait for input, some wires need to start with a value (as inputs to the entire system). The first section specifies these values. For example, x00: 1 means that the wire named x00 starts with the value 1 (as if a gate is already outputting that value onto that wire).

The second section lists all of the gates and the wires connected to them. For example, x00 AND y00 -> z00 describes an instance of an AND gate which has wires x00 and y00 connected to its inputs and which will write its output to wire z00.

In this example, simulating these gates eventually causes 0 to appear on wire z00, 0 to appear on wire z01, and 1 to appear on wire z02.

Ultimately, the system is trying to produce a number by combining the bits on all wires starting with z. z00 is the least significant bit, then z01, then z02, and so on.

In this example, the three output bits form the binary number 100 which is equal to the decimal number 4.

Here's a larger example:

x00: 1
x01: 0
x02: 1
x03: 1
x04: 0
y00: 1
y01: 1
y02: 1
y03: 1
y04: 1

ntg XOR fgs -> mjb
y02 OR x01 -> tnw
kwq OR kpj -> z05
x00 OR x03 -> fst
tgd XOR rvg -> z01
vdt OR tnw -> bfw
bfw AND frj -> z10
ffh OR nrd -> bqk
y00 AND y03 -> djm
y03 OR y00 -> psh
bqk OR frj -> z08
tnw OR fst -> frj
gnj AND tgd -> z11
bfw XOR mjb -> z00
x03 OR x00 -> vdt
gnj AND wpb -> z02
x04 AND y00 -> kjc
djm OR pbm -> qhw
nrd AND vdt -> hwm
kjc AND fst -> rvg
y04 OR y02 -> fgs
y01 AND x02 -> pbm
ntg OR kjc -> kwq
psh XOR fgs -> tgd
qhw XOR tgd -> z09
pbm OR djm -> kpj
x03 XOR y03 -> ffh
x00 XOR y04 -> ntg
bfw OR bqk -> z06
nrd XOR fgs -> wpb
frj XOR qhw -> z04
bqk OR frj -> z07
y03 OR x01 -> nrd
hwm AND bqk -> z03
tgd XOR rvg -> z12
tnw OR pbm -> gnj
After waiting for values on all wires starting with z, the wires in this system have the following values:

bfw: 1
bqk: 1
djm: 1
ffh: 0
fgs: 1
frj: 1
fst: 1
gnj: 1
hwm: 1
kjc: 0
kpj: 1
kwq: 0
mjb: 1
nrd: 1
ntg: 0
pbm: 1
psh: 1
qhw: 1
rvg: 0
tgd: 0
tnw: 1
vdt: 1
wpb: 0
z00: 0
z01: 0
z02: 0
z03: 1
z04: 0
z05: 1
z06: 1
z07: 1
z08: 1
z09: 1
z10: 1
z11: 0
z12: 0
Combining the bits from all wires starting with z produces the binary number 0011111101000. Converting this number to decimal produces 2024.

Simulate the system of gates and wires. What decimal number does it output on the wires starting with z?

In [194]:
from functools import cached_property

class Gate:
    def __init__(self, name):
        self.name = name
        self.wait = -1
        self.parents = []
        self.children = set()
        self.op = None
        self.value = None


    def __lt__(self, other):
        return self.name < other.name

    def __repr__(self):
        return self.name

    def compute(self):
        if not self.op:
            return
        
        x = self.parents[0].value
        y = self.parents[1].value

        if self.op == 'AND':
            self.value = x and y
        
        elif self.op == 'OR':
            self.value = x or y

        elif self.op == 'XOR':
            self.value = 1 if (x != y) else 0

    # added for part 2:
    @cached_property
    def all_inputs(self):
        inputs = set()
        if self.name[0] == 'x' or self.name[0] == 'y':
            inputs.add(self)
            return inputs
        for p in self.parents:
            if p.name[0] == 'x' or p.name[0] == 'y':
                inputs.add(p)
            else:
                inputs.update(p.all_inputs)
        return inputs

    @cached_property
    def all_outputs(self):
        outputs = set()
        if self.name[0] == 'z':
            outputs.add(self)
            return outputs
        for c in self.children:
            if c.name[0] == 'z':
                outputs.add(c)
            else:
                outputs.update(c.all_outputs)
        return outputs



In [195]:
with open('input24.txt') as file:
    data = [l.rstrip() for l in file]

In [196]:
import re
start_state = []
gates = {}
for l in data:
    if ":" in l:
        gate, value = l.split(':')
        start_state.append((gate, int(value)))
    elif "->" in l:
        logic, out = re.split(r' -> ', l)
        in1, op, in2 = re.split(r' (AND|XOR|OR) ', logic)
        
        # Create and connect nodes:
        for inc in in1, in2:
            gates.setdefault(inc, Gate(inc))
        gates.setdefault(out, Gate(out))
        gates[out].op = op
        gates[out].parents = [gates[in1], gates[in2]]
        gates[out].wait = 2
        for inc in in1, in2:
            gates[inc].children.add(gates[out])

In [197]:
# init:
queue = []
for id, v in start_state:
    gates[id].value = v   
    for child in gates[id].children:
        child.wait -= 1
        if child.wait == 0:
            queue.append(child)

In [198]:
while queue:
    gate = queue.pop()
    gate.compute()
    for child in gate.children:
        child.wait -= 1
        if child.wait == 0:
            queue.append(child)

In [199]:
z_nodes = sorted([g for g in gates if g[0] == 'z'], reverse=True)
z_values = [gates[z].value for z in z_nodes]
int(''.join([str(z) for z in z_values]), 2)

45121475050728

Your puzzle answer was 45121475050728.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
After inspecting the monitoring device more closely, you determine that the system you're simulating is trying to add two binary numbers.

Specifically, it is treating the bits on wires starting with x as one binary number, treating the bits on wires starting with y as a second binary number, and then attempting to add those two numbers together. The output of this operation is produced as a binary number on the wires starting with z. (In all three cases, wire 00 is the least significant bit, then 01, then 02, and so on.)

The initial values for the wires in your puzzle input represent just one instance of a pair of numbers that sum to the wrong value. Ultimately, any two binary numbers provided as input should be handled correctly. That is, for any combination of bits on wires starting with x and wires starting with y, the sum of the two numbers those bits represent should be produced as a binary number on the wires starting with z.

For example, if you have an addition system with four x wires, four y wires, and five z wires, you should be able to supply any four-bit number on the x wires, any four-bit number on the y numbers, and eventually find the sum of those two numbers as a five-bit number on the z wires. One of the many ways you could provide numbers to such a system would be to pass 11 on the x wires (1011 in binary) and 13 on the y wires (1101 in binary):

x00: 1
x01: 1
x02: 0
x03: 1
y00: 1
y01: 0
y02: 1
y03: 1
If the system were working correctly, then after all gates are finished processing, you should find 24 (11+13) on the z wires as the five-bit binary number 11000:

z00: 0
z01: 0
z02: 0
z03: 1
z04: 1
Unfortunately, your actual system needs to add numbers with many more bits and therefore has many more wires.

Based on forensic analysis of scuff marks and scratches on the device, you can tell that there are exactly four pairs of gates whose output wires have been swapped. (A gate can only be in at most one such pair; no gate's output was swapped multiple times.)

For example, the system below is supposed to find the bitwise AND of the six-bit number on x00 through x05 and the six-bit number on y00 through y05 and then write the result as a six-bit number on z00 through z05:

x00: 0
x01: 1
x02: 0
x03: 1
x04: 0
x05: 1
y00: 0
y01: 0
y02: 1
y03: 1
y04: 0
y05: 1

x00 AND y00 -> z05
x01 AND y01 -> z02
x02 AND y02 -> z01
x03 AND y03 -> z03
x04 AND y04 -> z04
x05 AND y05 -> z00
However, in this example, two pairs of gates have had their output wires swapped, causing the system to produce wrong answers. The first pair of gates with swapped outputs is x00 AND y00 -> z05 and x05 AND y05 -> z00; the second pair of gates is x01 AND y01 -> z02 and x02 AND y02 -> z01. Correcting these two swaps results in this system that works as intended for any set of initial values on wires that start with x or y:

x00 AND y00 -> z00
x01 AND y01 -> z01
x02 AND y02 -> z02
x03 AND y03 -> z03
x04 AND y04 -> z04
x05 AND y05 -> z05
In this example, two pairs of gates have outputs that are involved in a swap. By sorting their output wires' names and joining them with commas, the list of wires involved in swaps is z00,z01,z02,z05.

Of course, your actual system is much more complex than this, and the gates that need their outputs swapped could be anywhere, not just attached to a wire starting with z. If you were to determine that you need to swap output wires aaa with eee, ooo with z99, bbb with ccc, and aoc with z24, your answer would be aaa,aoc,bbb,ccc,eee,ooo,z24,z99.

Your system of gates and wires has four pairs of gates which need their output wires swapped - eight wires in total. Determine which four pairs of gates need their outputs swapped so that your system correctly performs addition; what do you get if you sort the names of the eight wires involved in a swap and then join those names with commas?



In [200]:
"""
So the wiring is supposed to produce a binary adder. This means, that we know what the supposed wiring between x, y and z gates looks like:
z00 is x00 XOR y00
carry00 is x00 AND y00
...and so on

I believe our job is to
(1) create a function that analyses the total wiring and identifies which, if any input/output pairs are wired incorrectly

(2) from the set of incorrect wiring pairs, we can probably just try all combinations of swaps to identify a working adder (it is also possible that the swaps involve some wirings that are already working, and that the swaps will zero out for them). For that we will also need to identify all upstream/downstream nodes for a given in/output

For (1), that job is comparatively easy when it comes to the first bits. When it comes to higher level bits, it will be easy to say that *something* is wrong, but much harder to pinpoint the failure algorithmically

Perhaps an easier approach is not to map the logic of the whole algorithm exhaustively, but to write a simulator where we can compare in- and outputs easily and then to probe with numbers with mostly zeros to see what works and what doesn't
"""

"\nSo the wiring is supposed to produce a binary adder. This means, that we know what the supposed wiring between x, y and z gates looks like:\nz00 is x00 XOR y00\ncarry00 is x00 AND y00\n...and so on\n\nI believe our job is to\n(1) create a function that analyses the total wiring and identifies which, if any input/output pairs are wired incorrectly\n\n(2) from the set of incorrect wiring pairs, we can probably just try all combinations of swaps to identify a working adder (it is also possible that the swaps involve some wirings that are already working, and that the swaps will zero out for them). For that we will also need to identify all upstream/downstream nodes for a given in/output\n\nFor (1), that job is comparatively easy when it comes to the first bits. When it comes to higher level bits, it will be easy to say that *something* is wrong, but much harder to pinpoint the failure algorithmically\n\nPerhaps an easier approach is not to map the logic of the whole algorithm exhaustiv

In [201]:
def num_to_gates(num):
    return [int(c) for c in reversed(f'{num:045b}')]

def gates_to_num(gates):
    z_nodes = sorted([g for g in gates.values() if g.name[0] == 'z'], reverse=True)
    z_values = [g.value for g in z_nodes]
    return int(''.join([str(z) for z in z_values]), 2)    

def seed(gate, value, queue):
    gate.value = value
    gate.wait = -1
    for child in gate.children:
        child.wait -= 1
        if child.wait == 0:
            queue.append(child)


def test_wiring(gates, x, y):
    #reset network:
    for g in gates.values():
        g.value = 0
        g.wait = 2

    # initial values:
    queue = []
    for ix, v in enumerate(num_to_gates(x)):
        name = f'x{ix:02}'
        seed(gates[name], v, queue)

    for ix, v in enumerate(num_to_gates(y)):
        name = f'y{ix:02}'
        seed(gates[name], v, queue)

    # compute:
    while queue:
        gate = queue.pop()
        gate.compute()
        for child in gate.children:
            child.wait -= 1
            if child.wait == 0:
                queue.append(child)

    # read out:
    return gates_to_num(gates)

In [202]:
# Having looked at some simple cases, the calculator seems to be doing alright a lot of the time
bin(test_wiring(gates, 0b100000, 0b100000))

'0b1000000'

In [203]:
# Let's automate iterating through simple cases:

# Single 1 bit inputs:
import random
x = 1
for ix in range(45):
    if (z := test_wiring(gates, x, 0)) != x:
        print(f'{x:} + 0 resulted in \n{z}')

    if (z := test_wiring(gates, 0, x)) != x:
        print(f'0 + {x:} resulted in \n    {z}')

    x <<= 1
len(bin(x)[2:])


1024 + 0 resulted in 
2048
0 + 1024 resulted in 
    2048
262144 + 0 resulted in 
524288
0 + 262144 resulted in 
    524288
16777216 + 0 resulted in 
33554432
0 + 16777216 resulted in 
    33554432
8589934592 + 0 resulted in 
17179869184
0 + 8589934592 resulted in 
    17179869184


46

In [204]:
# Single input below 0b1000000000 (first error in single bit test): No errors

for _ in range(1000):
    x = random.randint(0,0b1000000000)
    if (z := test_wiring(gates, x, 0)) != x:
        print(f'{x:b} + 0 resulted in \n{z:b}')
    if (z := test_wiring(gates, 0, x)) != x:
        print(f'0 + {x:b} resulted in \n    {z:b}')

In [205]:
# Double input below 0b1000000000: No errors
for _ in range(1000):
    x = random.randint(0,0b1000000000)
    y = random.randint(0,0b1000000000)
    if (z := test_wiring(gates, x, y)) != x + y:
        print(f'{x:b} {y:b} resulted in {z:b}')

In [206]:
# let's write a function that will, for every position, test (x: 1, y:0), (x: 0, y: 1), (x: 1, y: 1) and report what the error was at each bit

def print_errors(out, corr):
        output = num_to_gates(out)
        ground_truth = num_to_gates(corr)
        for jx in range(45):
            if output[jx] != ground_truth[jx]:
                print(f'The output at position {jx} was {output[jx]} (should have been {ground_truth[jx]})')


one_bit = 1
for ix in range(45):
    if (z := test_wiring(gates, one_bit, 0)) != one_bit:
        print(f'\nFor inputs x=1, y=0 at position {ix} an error occured:')
        print_errors(z, one_bit)

    if (z := test_wiring(gates, 0, one_bit)) != one_bit:
        print(f'\nFor inputs x=0, y=1 at position {ix} an error occured:')
        print_errors(z, one_bit)

    if (z := test_wiring(gates, one_bit, one_bit)) != 2 * one_bit:
        print(f'\nFor inputs x=1, y=1 at position {ix} an error occured:')
        print_errors(z, 2 * one_bit)

    one_bit <<= 1


For inputs x=1, y=1 at position 9 an error occured:
The output at position 10 was 0 (should have been 1)
The output at position 11 was 1 (should have been 0)

For inputs x=1, y=0 at position 10 an error occured:
The output at position 10 was 0 (should have been 1)
The output at position 11 was 1 (should have been 0)

For inputs x=0, y=1 at position 10 an error occured:
The output at position 10 was 0 (should have been 1)
The output at position 11 was 1 (should have been 0)

For inputs x=1, y=1 at position 10 an error occured:
The output at position 10 was 1 (should have been 0)
The output at position 11 was 0 (should have been 1)

For inputs x=1, y=1 at position 17 an error occured:
The output at position 18 was 0 (should have been 1)
The output at position 19 was 1 (should have been 0)

For inputs x=1, y=0 at position 18 an error occured:
The output at position 18 was 0 (should have been 1)
The output at position 19 was 1 (should have been 0)

For inputs x=0, y=1 at position 18 an er

In [207]:
"""
Ok. So our errors are as follows:
- The adder for z 10 outputs to z 11; but the carry from idx 10 gets applied to z10 instead of z11
- The adder for z18 outputs at z19 with the same carry problem: carry from idx 18 gets applied to z18

- The adder for idx 24 falsely outputs at idx 25; carry at idx 24 falsely gets applied to z24. HOWEVER, the carry from idx23 correctly moves to z24

- The adder for idx 33 outputs at idx 34; HOWEVER, carry from idx 33 correctly moves on to z34
"""

'\nOk. So our errors are as follows:\n- The adder for z 10 outputs to z 11; but the carry from idx 10 gets applied to z10 instead of z11\n- The adder for z18 outputs at z19 with the same carry problem: carry from idx 18 gets applied to z18\n\n- The adder for idx 24 falsely outputs at idx 25; carry at idx 24 falsely gets applied to z24. HOWEVER, the carry from idx23 correctly moves to z24\n\n- The adder for idx 33 outputs at idx 34; HOWEVER, carry from idx 33 correctly moves on to z34\n'

In [208]:
# Editorial remark: After this, I did some more analysis with probing inputs that in retrospect yielded little. I leave it in the notebook to show my process with a problem I haven't dealt with in the past

In [209]:
# Let's see if anything changes if we input '11' blocks (eg 000000...1100):

bit_11 = 0b11
for ix in range(44):

    if (z := test_wiring(gates, bit_11, bit_11)) != 2 * bit_11:
        print(f'\nFor inputs x=11, y=11 at positions {ix} and {ix + 1} an error occured:')
        print_errors(z, 2 * bit_11)

    bit_11 <<= 1


For inputs x=11, y=11 at positions 8 and 9 an error occured:
The output at position 10 was 0 (should have been 1)
The output at position 11 was 1 (should have been 0)

For inputs x=11, y=11 at positions 10 and 11 an error occured:
The output at position 10 was 1 (should have been 0)
The output at position 11 was 0 (should have been 1)

For inputs x=11, y=11 at positions 16 and 17 an error occured:
The output at position 18 was 0 (should have been 1)
The output at position 19 was 1 (should have been 0)

For inputs x=11, y=11 at positions 18 and 19 an error occured:
The output at position 18 was 1 (should have been 0)
The output at position 19 was 0 (should have been 1)

For inputs x=11, y=11 at positions 23 and 24 an error occured:
The output at position 24 was 0 (should have been 1)

For inputs x=11, y=11 at positions 24 and 25 an error occured:
The output at position 24 was 1 (should have been 0)
The output at position 25 was 0 (should have been 1)

For inputs x=11, y=11 at positions

In [210]:
"""
What does this tell us?
In the 11 at ix 9/10 case we get no errors. Why? At idx 10 we have as inputs 1 from the carry, 1 from x and 1 from y. So 1 should go to z10 and 1 to the carry. The wiring switches these outputs, but that's no problem since both are 1. Therefore no errors. 
In the other cases, errors don't cancel out and we get faulty behavior as in the earlier test.

The ix18-adder shows the same behavior as the ix10-adder

The ix24-adder is a bit different: 
- carry from ix23 correctly applies to z24 (no errors in the 22/23 case)
- in the 23/24 case, we get a single error at z24. There should have been a 1 at idx 24 (3 1 inputs) and a 1 at idx 25 (from the carry). In practice, we get a 0 at z24 and a (correct) 1 at z25. I want to do more testing at this node to suss out exactly what is happening here.

For the ix33-adder behavior is also a bit unclear and warrants further testing.
"""

"\nWhat does this tell us?\nIn the 11 at ix 9/10 case we get no errors. Why? At idx 10 we have as inputs 1 from the carry, 1 from x and 1 from y. So 1 should go to z10 and 1 to the carry. The wiring switches these outputs, but that's no problem since both are 1. Therefore no errors. \nIn the other cases, errors don't cancel out and we get faulty behavior as in the earlier test.\n\nThe ix18-adder shows the same behavior as the ix10-adder\n\nThe ix24-adder is a bit different: \n- carry from ix23 correctly applies to z24 (no errors in the 22/23 case)\n- in the 23/24 case, we get a single error at z24. There should have been a 1 at idx 24 (3 1 inputs) and a 1 at idx 25 (from the carry). In practice, we get a 0 at z24 and a (correct) 1 at z25. I want to do more testing at this node to suss out exactly what is happening here.\n\nFor the ix33-adder behavior is also a bit unclear and warrants further testing.\n"

In [211]:
# ix-24 adder testing (I rather like the formatting I came up with, even though I should have switched to different approaches by this point):

# Scenarios: carry: 0/1, x24+y24: 0/1/2, x25+y25: 0/1

def print_at(num, corr, from_to):
    ground_truth = num_to_gates(corr)
    num = num_to_gates(num)
    for i in from_to:
        if num[i] == ground_truth[i]:
            print(f'{num[i]}   ', end='')
        else:
            print(f'\033[31m{num[i]}\033[0m   ', end='')

carry_23 = 1 << 23
print('Inputs:                      Output: 23, 24, 25, 26')
for carry_23 in range(2):
    for xy_24 in range(3):
        for xy_25 in range(2):
            
            x = y = 0
            if carry_23:
                x = y = 1 << 23
                
            if xy_24 == 1:
                if test_wiring(gates, x + (1 << 24), y) != test_wiring(gates, x, y + (1 << 24)):
                    print('Error! Asymmetric handling outcome for x/y switch!')
                x += 1 << 24
            elif xy_24 == 2:
                x += 1 << 24
                y += 1 << 24
            
            if xy_25:
                if test_wiring(gates, x + (1 << 25), y) != test_wiring(gates, x, y + (1 << 25)):
                    print('Error! Asymmetric handling outcome for x/y switch!')
                x += 1 << 25

            print(f'23-Carry: {carry_23}, x24+y24: {xy_24}, x25: {xy_25}:      ', end='')
            print_at(test_wiring(gates, x, y), x+y, range(23, 27))
            print()

Inputs:                      Output: 23, 24, 25, 26
23-Carry: 0, x24+y24: 0, x25: 0:      0   0   0   0   
23-Carry: 0, x24+y24: 0, x25: 1:      0   0   1   0   
23-Carry: 0, x24+y24: 1, x25: 0:      0   0   1   0   
23-Carry: 0, x24+y24: 1, x25: 1:      0   0   0   1   
23-Carry: 0, x24+y24: 2, x25: 0:      0   1   0   0   
23-Carry: 0, x24+y24: 2, x25: 1:      0   1   1   0   
23-Carry: 1, x24+y24: 0, x25: 0:      0   1   0   0   
23-Carry: 1, x24+y24: 0, x25: 1:      0   1   1   0   
23-Carry: 1, x24+y24: 1, x25: 0:      0   1   1   0   
23-Carry: 1, x24+y24: 1, x25: 1:      0   1   0   1   
23-Carry: 1, x24+y24: 2, x25: 0:      0   0   1   0   
23-Carry: 1, x24+y24: 2, x25: 1:      0   0   0   1   


In [212]:
""" Analysis:
No 23carry:
xy24 = 1 -> 1 applied to carry instead of z24
xy25 = 2 -> 1 applied to z24 instead of carry

With 23carry:
xy24 = 0 -> correct behavior
xy24 = 1 -> 23carry applied to z24 AND xy24 applied to carry
xy24 = 2 -> 23carry applied to z24 AND 24carry applied to z24 (therefore z24 == 0). However, z25 *also* gets a carry-1, I don't know from where
"""

" Analysis:\nNo 23carry:\nxy24 = 1 -> 1 applied to carry instead of z24\nxy25 = 2 -> 1 applied to z24 instead of carry\n\nWith 23carry:\nxy24 = 0 -> correct behavior\nxy24 = 1 -> 23carry applied to z24 AND xy24 applied to carry\nxy24 = 2 -> 23carry applied to z24 AND 24carry applied to z24 (therefore z24 == 0). However, z25 *also* gets a carry-1, I don't know from where\n"

In [213]:
# idx 33 adder testing:
carry_32 = 1 << 32
print('Inputs:                      Output: 32, 33, 34, 35')
for carry_32 in range(2):
    for xy_33 in range(3):
        for xy_34 in range(2):
            
            x = y = 0
            if carry_32:
                x = y = 1 << 32
                
            if xy_33 == 1:
                if test_wiring(gates, x + (1 << 33), y) != test_wiring(gates, x, y + (1 << 33)):
                    print('Error! Asymmetric handling outcome for x/y switch!')
                x += 1 << 33
            elif xy_33 == 2:
                x += 1 << 33
                y += 1 << 33
            
            if xy_34:
                if test_wiring(gates, x + (1 << 34), y) != test_wiring(gates, x, y + (1 << 34)):
                    print('Error! Asymmetric handling outcome for x/y switch!')
                x += 1 << 34

            print(f'32-Carry: {carry_32}, x33+y33: {xy_33}, x34: {xy_34}:      ', end='')
            print_at(test_wiring(gates, x, y), x+y, range(32, 36))
            print()

Inputs:                      Output: 32, 33, 34, 35
32-Carry: 0, x33+y33: 0, x34: 0:      0   0   0   0   
32-Carry: 0, x33+y33: 0, x34: 1:      0   0   1   0   
32-Carry: 0, x33+y33: 1, x34: 0:      0   0   1   0   
32-Carry: 0, x33+y33: 1, x34: 1:      0   0   0   1   
32-Carry: 0, x33+y33: 2, x34: 0:      0   0   1   0   
32-Carry: 0, x33+y33: 2, x34: 1:      0   0   0   1   
32-Carry: 1, x33+y33: 0, x34: 0:      0   0   1   0   
32-Carry: 1, x33+y33: 0, x34: 1:      0   0   0   1   
32-Carry: 1, x33+y33: 1, x34: 0:      0   1   0   0   
32-Carry: 1, x33+y33: 1, x34: 1:      0   1   1   0   
32-Carry: 1, x33+y33: 2, x34: 0:      0   0   1   0   
32-Carry: 1, x33+y33: 2, x34: 1:      0   0   0   1   


In [214]:
"""
Analysis:
No 32-carry:
xy33 = 1 -> 1 applied to carry instead of z33
xy33 = 2 -> correct behavior

With 32carry:
xy33 = 0 -> 32carry incorrectly applied to z34 instead of z33
xy33 = 1 -> 1 0 instead of 0 1
xy24 = 2 -> 0 1 instead of 1 1

At the end of the day, the behavior of the 24 and 33 adders cannot be simply described as ones getting routed to the wrong output. The behavior seems more complex than that. 
"""

'\nAnalysis:\nNo 32-carry:\nxy33 = 1 -> 1 applied to carry instead of z33\nxy33 = 2 -> correct behavior\n\nWith 32carry:\nxy33 = 0 -> 32carry incorrectly applied to z34 instead of z33\nxy33 = 1 -> 1 0 instead of 0 1\nxy24 = 2 -> 0 1 instead of 1 1\n\nAt the end of the day, the behavior of the 24 and 33 adders cannot be simply described as ones getting routed to the wrong output. The behavior seems more complex than that. \n'

In [215]:
# Let's look at graphs (ed: finally! Having exhausted all other options, I do what I am afraid of. In retrospect, it was surprisingly pleasant and not as complicated as I feared):

queue = []
visited = set()

for level in range(38):
    queue.append(gates['x' + f'{level:02}'])
    queue.append(gates['y' + f'{level:02}'])

    new_queue = []
    for g in queue:
        if g.parents:
            print(f'{g.parents[0]} {g.op} {g.parents[1]} -> {g.name}')
        for c in g.children:
            if c not in visited:
                new_queue.append(c)
                visited.add(c)
    queue = new_queue

x00 XOR y00 -> z00
x00 AND y00 -> hmc
brk XOR hmc -> z01
hmc AND brk -> pjk
y01 AND x01 -> vpj
x01 XOR y01 -> brk
pjk OR vpj -> rhd
y02 AND x02 -> wwc
x02 XOR y02 -> rcm
rcm AND rhd -> qvh
rhd XOR rcm -> z02
qvh OR wwc -> sgb
y03 AND x03 -> vpp
x03 XOR y03 -> psb
sgb AND psb -> tjj
psb XOR sgb -> z03
tjj OR vpp -> jhh
y04 XOR x04 -> vws
y04 AND x04 -> bbw
jhh XOR vws -> z04
vws AND jhh -> shg
shg OR bbw -> ckj
y05 XOR x05 -> tbw
y05 AND x05 -> hnw
ckj AND tbw -> nth
ckj XOR tbw -> z05
hnw OR nth -> phh
x06 XOR y06 -> wtm
y06 AND x06 -> mgc
phh AND wtm -> cjd
wtm XOR phh -> z06
mgc OR cjd -> std
y07 XOR x07 -> jfr
y07 AND x07 -> nsg
jfr AND std -> tdd
std XOR jfr -> z07
tdd OR nsg -> cct
x08 XOR y08 -> cjn
y08 AND x08 -> gdm
cct XOR cjn -> z08
cjn AND cct -> kqg
gdm OR kqg -> wpc
x09 XOR y09 -> rds
x09 AND y09 -> bmn
rds AND wpc -> cmj
wpc XOR rds -> z09
bmn OR cmj -> mwq
x10 AND y10 -> kqf
y10 XOR x10 -> sqr
mwq AND sqr -> kgd
sqr XOR mwq -> mwk
kgd OR kqf -> z10
y11 AND x11 -> pjp
x11

In [216]:
# Looking at the first pairs, the adder logic works like this:
# x00 XOR y00 -> z00
# x00 AND y00 -> carry00 (hmc)
# (x01 XOR y01 [brk]) XOR carry00 -> z01
# ((x01 XOR y01 [brk]) AND carry00[hmc])[pjk] OR (x01 AND y01)[vpj] -> carry01 [rhd]
# (y02 XOR x02)[rcm] XOR carry01 -> z02
# ((x02 XOR y02)[rcm] AND carry01[rhd])[qvh] OR (x02 AND y02)[wwc] -> carry03[sgb]

# So the input for zi should be (xi XOR yi) XOR carry(i-1)
# The input for the carry is ((xi XOR yi) AND carry(i-1)) OR (xi AND yi)

In [217]:
# What's the situation for input/outputs at ix 10 and 11?
# carry09: mwq
# {(y10 XOR x10)[sqr] AND carry09[mwq])}[kgd] OR y10 AND x10[kqf] -> z10 (should be carry10!)
# (y10 XOR x10)[sqr] XOR carry09[mwq]) -> mwk (is supposed to be carry11 but has inputs for z10)

# Therefore: Switching z10 and mwk should do the trick.

In [218]:
# What about ix 24 and 25?
# carry23: mrs
# (y24 AND x24)[jmh] XOR carry23[mrs] -> z24
# ((y24 AND x24)[jmh] AND carry23[mrs])[bmv] OR (y24 XOR x24)[hsw] -> mtn
# So the swap is between jmh and hsw

# And 33 and 34 just for fun?
# carry32: cvt
# carry32[cvt] AND (y33 XOR x33)[wwp] -> z33
# (x33 AND y33)[wvp] OR {(y33 XOR x33)[wwp] XOR carry32[cvt]}[gqp] -> vtd
# Solution: Swap gqp with z33 

In [219]:
"""
(Ed: In reality, I tried the trial and error approach of just switching gates around the problem outputs before I looked at graphs. However, due to some bugs in my algorithm it did not come up with a solution. Once I looked at graphs, I realized the simple switch MUST be possible and fixed the bugs. At the same time, the graph analysis also yielded the required gate swaps directly. So I ended up with two ways to solve the problem)

I want to switch gears a little now. What we will do is this: We have isolated the z-gates showing problematic behavior, and they come in twos: z10 and z11, z18 and z19, z24 and z25 and z33 and z34. Since we can do four swaps, presumably a single swap between two gates will allow us to fix their behavior (it is also possible that some middle gates involving many more gates z-gates are part of the correct swap. This would be much more complex, and for now I will bet that we can look at each pair in isolation.). We will find all gates that pertain to that pair of outputs, preferably those that *only* affect them, and see if we can find, through permutation, a swap that fixes the issues.
"""

'\n(Ed: In reality, I tried the trial and error approach of just switching gates around the problem outputs before I looked at graphs. However, due to some bugs in my algorithm it did not come up with a solution. Once I looked at graphs, I realized the simple switch MUST be possible and fixed the bugs. At the same time, the graph analysis also yielded the required gate swaps directly. So I ended up with two ways to solve the problem)\n\nI want to switch gears a little now. What we will do is this: We have isolated the z-gates showing problematic behavior, and they come in twos: z10 and z11, z18 and z19, z24 and z25 and z33 and z34. Since we can do four swaps, presumably a single swap between two gates will allow us to fix their behavior (it is also possible that some middle gates involving many more gates z-gates are part of the correct swap. This would be much more complex, and for now I will bet that we can look at each pair in isolation.). We will find all gates that pertain to that

In [220]:
from itertools import combinations

def test_gates(gates, z_tocheck):
    z1, z2 = z_tocheck
    for carry_bellow in range(2):
        for xy_lower in range(3):
            for xy_higher in range(2):

                # Set inputs:
                x = y = 0
                if carry_bellow:
                    x = y = 1 << (z1 - 1)
                    
                if xy_lower == 1:
                    if test_wiring(gates, x + (1 << z1), y) != test_wiring(gates, x, y + (1 << z1)):
                        # print('symmetry failed')
                        return False
                    x += 1 << z1
                elif xy_lower == 2:
                    x += 1 << z1
                    y += 1 << z1
                
                if xy_higher:
                    if test_wiring(gates, x + (1 << z2), y) != test_wiring(gates, x, y + (1 << z2)):
                        # print('symmetry failed')
                        return False
                    x += 1 << z2

                # Test:
                if x + y != test_wiring(gates, x, y):
                    # print (f'{x=}, {y=} lead to outcome {test_wiring(gates, x, y)}')
                    return False
    return True
test_gates(gates, [11,12])

False

In [221]:
def swap_gates(z_tofix, swappable_gates, gates):
    for g1, g2 in combinations(swappable_gates.values(), 2):

        # swap gates:
        for c in g1.children:
            c.parents.remove(g1)
            c.parents.append(g2)

        for c in g2.children:
            c.parents.remove(g2)
            c.parents.append(g1)
        
        g1c, g2c = g1.children, g2.children
        g1.children = g2c
        g2.children = g1c

        g1n, g2n = g1.name, g2.name
        g1.name, g2.name = g2n, g1n
        
        if test_gates(gates, z_tofix):
            print(f'fixed by swapping Gates {g1} and {g2}')
            gates[g1.name] = g1
            gates[g2.name] = g2
            return ((g1, g2))
            
        # repair:
        g1.children = g1c
        g2.children = g2c
        g1.name, g2.name = g1n, g2n

        for c in g1.children:
            c.parents.remove(g2)
            c.parents.append(g1)

        for c in g2.children:
            c.parents.remove(g1)
            c.parents.append(g2)

In [222]:
from copy import deepcopy

mod_gates = deepcopy(gates)

# First pair:
z_tofix = [10,11]

def get_affected_gates(z_tofix, gates, mode='min'):
    z1, z2 = z_tofix
    outputs_affected = range(z1, z2+2)

    gates_aff = {k:v for k, v in mod_gates.items() if any(mod_gates['z' + f'{ix:02}'] in v.all_outputs for ix in outputs_affected)}

    # excluding gates that affect lower level behavior (where we saw no errors):
    if mode == 'min':
        names_lower = ['z' + f'{i:02}' for i in range(z_tofix[0] - 1)]
        gates_aff = {k:v for k, v in gates_aff.items() if all(str(o) not in names_lower for o in v.all_outputs)}
    
    return gates_aff
swapped_gates = []
swapped_gates.extend(swap_gates(z_tofix, get_affected_gates(z_tofix, mod_gates), mod_gates))

fixed by swapping Gates mwk and z10


In [223]:
# Moving on to z18 and z19:
z_tofix = [18,19]
swapped_gates.extend(swap_gates(z_tofix, get_affected_gates(z_tofix, mod_gates), mod_gates))

fixed by swapping Gates z18 and qgd


In [224]:
# Z 24 and 25:
z_tofix = [24,25]
swapped_gates.extend(swap_gates(z_tofix, get_affected_gates(z_tofix, mod_gates), mod_gates))

fixed by swapping Gates jmh and hsw


In [225]:
# z33 and z34
z_tofix = [33,34]
swapped_gates.extend(swap_gates(z_tofix, get_affected_gates(z_tofix, mod_gates), mod_gates))

fixed by swapping Gates z33 and gqp


In [226]:
','.join(g.name for g in sorted(swapped_gates))

'gqp,hsw,jmh,mwk,qgd,z10,z18,z33'

That's the right answer! You are one gold star closer to finding the Chief Historian.